# Gas Analyzer — Data Exploration

This notebook walks through loading, validating, cleaning, and visualising
gas measurement data using the Gas Analyzer package.


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

from src.ingestion.loader import DataLoader
from src.processing.cleaner import DataCleaner
from src.processing.validator import DataValidator
from src.visualization.plotter import GasPlotter
from src.utils.logger import setup_logger

setup_logger('INFO')
print('Gas Analyzer loaded.')

## 1. Load Sample Data

In [ ]:
data = DataLoader.load_sample()
print(f'Shape: {data.shape}')
data.head()

## 2. Validate

In [ ]:
validator = DataValidator('../config/settings.yaml')
report = validator.validate(data)
print(report.summary())

## 3. Clean

In [ ]:
cleaner = DataCleaner(
    outlier_method='iqr',
    normalization='none',
    resampling_freq='1h'
)
data_clean = cleaner.fit_transform(data)
print(f'After cleaning: {data_clean.shape}')

## 4. Visualise Time Series

In [ ]:
plotter = GasPlotter('../outputs/plots')
fig = plotter.plot_time_series(
    data_clean,
    columns=['CH4', 'N2', 'CO2', 'pressure_bar', 'flow_rate_m3h'],
    title='Gas Measurements — 2024'
)
plt.show()

## 5. Correlation Heatmap

In [ ]:
from src.models.correlation import CorrelationAnalyzer

corr_analyzer = CorrelationAnalyzer(method='pearson')
corr_matrix = corr_analyzer.correlation_matrix(data_clean)

fig = plotter.plot_correlation_heatmap(corr_matrix)
plt.show()

## 6. Anomaly Detection

In [ ]:
from src.models.anomaly import AnomalyDetector

detector = AnomalyDetector(method='isolation_forest', contamination=0.03)
anomalies = detector.fit_predict(data_clean)
print(f'Anomalies detected: {anomalies.sum()} ({100*anomalies.mean():.1f}%)')

fig = plotter.plot_anomalies(data_clean, anomalies, column='CH4')
plt.show()

## 7. Seasonal Decomposition

In [ ]:
from src.models.analyzer import GasAnalyzer

analyzer = GasAnalyzer(decomposition_period=24)
report = analyzer.analyze(data_clean, columns=['CH4', 'flow_rate_m3h'])

fig = plotter.plot_decomposition(report.decompositions['CH4'])
plt.show()

## 8. Descriptive Statistics

In [ ]:
report.descriptive_stats.style.background_gradient(cmap='coolwarm', axis=0)